In [19]:
from typing import TypedDict
from llm.llm import llm_model
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field


In [20]:
model = llm_model()

In [21]:
# 1. Define a Pydantic schema so the LLM returns a clean float (1.0 to 5.0)
class BlogEvaluation(BaseModel):
    score: float = Field(
        ...,
        ge=1.0,
        le=5.0,
        description="Rating from 1.0 to 5.0 based on how well the blog follows the outline and overall writing quality."
    )

In [22]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    evalution: float

In [23]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title 
    title = state['title']

    # call llm gen outline 
    promt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(promt).content

    # update state
    state['outline'] = outline

    return state



In [24]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [25]:
def evaluate_blog(state: BlogState) -> dict:
    outline = state['outline']
    content = state['content']

    prompt = (
        "Evaluate the following blog post on a scale of 1.0 to 5.0 based on "
        "how thoroughly and accurately it follows the provided outline.\n\n"
        f"OUTLINE:\n{outline}\n\n"
        f"BLOG CONTENT:\n{content}"
    )

    # Bind structured output so response.score is guaranteed to be a float
    structured_llm = model.with_structured_output(BlogEvaluation)
    response = structured_llm.invoke(prompt)

    return {'evalution': response.score}

In [27]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)


workflow = graph.compile()




In [28]:
# execution
initial_state = {'title': 'Rise of AI in the world'}

final_state = workflow.invoke(initial_state)
print("Evalution Score:", final_state['evalution'])
print(final_state)

Evalution Score: 4.5
{'title': 'Rise of AI in the world', 'outline': 'I. Introduction\n   A. Definition of AI\n   B. Brief history of AI\n   C. Importance of AI in the world\n\nII. AI in Healthcare\n   A. Diagnosis and treatment\n   B. Drug discovery\n   C. Personalized medicine\n   D. Remote patient monitoring\n\nIII. AI in Finance\n   A. Fraud detection\n   B. Investment analysis\n   C. Trading strategies\n   D. Risk management\n\nIV. AI in Transportation\n   A. Autonomous vehicles\n   B. Traffic management\n   C. Logistics optimization\n   D. Ride-sharing services\n\nV. AI in Retail\n   A. Personalized shopping\n   B. Inventory management\n   C. Customer service\n   D. Product recommendations\n\nVI. AI in Education\n   A. Adaptive learning\n   B. Personalized instruction\n   C. Gamification\n   D. Assessment and feedback\n\nVII. AI in Manufacturing\n   A. Predictive maintenance\n   B. Quality control\n   C. Supply chain optimization\n   D. Robotics\n\nVIII. AI in Energy\n   A. Renew

In [ ]:
print(final_state['content'])

**The Rise of Artificial Intelligence: A Global Revolution**

**Introduction:**
The rise of artificial intelligence (AI) is a phenomenon that has been unfolding over the past few decades, transforming the way we live, work, and interact with technology. This blog aims to provide a detailed look at the impact of AI on industries, society, and the future of AI, encouraging readers to embrace this transformative technology responsibly.

**Section 1: The Impact of AI on Industries**

**Healthcare**
- **AI in diagnostics**: AI algorithms can analyze medical images and patient data to identify diseases earlier and with greater accuracy than human doctors.
- **AI in drug discovery**: AI can accelerate the drug discovery process by predicting the efficacy and side effects of new compounds, reducing the time and cost of bringing new drugs to market.
- **AI in personalized medicine**: AI can tailor treatments to individual patients based on their genetic makeup, medical history, and lifestyle, l